# SupportIQ — Stage 1.1: Load & Inspect Bitext Dataset

> **Goal:** Ingest the raw Bitext customer support dataset from Hugging Face, pin its exact revision, save untouched raw artifacts to `data/raw/`, record comprehensive provenance in `METADATA.json`, and inspect initial columns, dtypes, and samples.
> Reference: `AGENT_GUIDE_v2.md` Stage 1.1.


### 1. Dependencies & Setup
Load required libraries for dataset loading, hashing, and tabular analysis.

In [1]:
import hashlib
import json
from datetime import UTC, datetime
from pathlib import Path

import polars as pl
from datasets import load_dataset
from huggingface_hub import HfApi

print(f"Polars version: {pl.__version__}")

Polars version: 1.44.2


### 2. Configuration & Paths
Define source repository, target raw directory, and artifact filenames.

In [2]:
DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
RAW_DIR = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

RAW_PARQUET_PATH = RAW_DIR / "bitext_raw.parquet"
RAW_CSV_PATH = RAW_DIR / "bitext_raw.csv"
METADATA_PATH = RAW_DIR / "METADATA.json"

print(f"Raw target directory: {RAW_DIR.resolve()}")

Raw target directory: /home/dvinix/Projects/supportiq/data/raw


### 3. Retrieve Repository Metadata & Revision SHA
Pin the exact commit revision SHA from Hugging Face Hub to guarantee reproducibility.

In [3]:
api = HfApi()
repo_info = api.dataset_info(DATASET_NAME)
revision_sha = repo_info.sha
card_data = repo_info.card_data or {}
license_name = getattr(card_data, "license", "cdla-sharing-1.0")

print(f"Dataset: {DATASET_NAME}")
print(f"Revision SHA: {revision_sha}")
print(f"License: {license_name}")

Dataset: bitext/Bitext-customer-support-llm-chatbot-training-dataset
Revision SHA: 430d1a89bd93bd1fa23c16f29dd53e73f0087443
License: cdla-sharing-1.0


### 4. Ingest Raw Dataset from Hugging Face
Load the dataset split directly using the Hugging Face `datasets` library.

In [4]:
ds = load_dataset(DATASET_NAME, split="train", revision=revision_sha)
print(f"Loaded {len(ds):,} raw records.")
print(f"Feature schema: {ds.features}")

Loaded 26,872 raw records.
Feature schema: {'flags': Value('string'), 'instruction': Value('string'), 'category': Value('string'), 'intent': Value('string'), 'response': Value('string')}


### 5. Save Raw Dataset Untouched
Save the raw dataset to `data/raw/` in both Parquet and CSV formats to preserve raw bits.

In [5]:
# Convert to Polars DataFrame
df = pl.from_arrow(ds.data.table)

# Write untouched raw parquet and csv
df.write_parquet(RAW_PARQUET_PATH)
df.write_csv(RAW_CSV_PATH)

parquet_bytes = RAW_PARQUET_PATH.stat().st_size
csv_bytes = RAW_CSV_PATH.stat().st_size
print(f"Saved {RAW_PARQUET_PATH.name} ({parquet_bytes / 1024 / 1024:.2f} MB)")
print(f"Saved {RAW_CSV_PATH.name} ({csv_bytes / 1024 / 1024:.2f} MB)")

Saved bitext_raw.parquet (3.04 MB)
Saved bitext_raw.csv (18.31 MB)


### 6. Compute Checksum & Write METADATA.json
Generate SHA-256 hashes of the raw files and record full dataset provenance.

In [6]:
def sha256_file(filepath: Path) -> str:
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()


parquet_sha256 = sha256_file(RAW_PARQUET_PATH)
csv_sha256 = sha256_file(RAW_CSV_PATH)

metadata = {
    "dataset_name": DATASET_NAME,
    "source_url": f"https://huggingface.co/datasets/{DATASET_NAME}",
    "revision_sha": revision_sha,
    "license": str(license_name),
    "download_timestamp_utc": datetime.now(UTC).isoformat(),
    "row_count": df.height,
    "column_count": df.width,
    "columns": df.columns,
    "artifacts": {
        "bitext_raw.parquet": {"sha256": parquet_sha256, "size_bytes": parquet_bytes},
        "bitext_raw.csv": {"sha256": csv_sha256, "size_bytes": csv_bytes},
    },
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("METADATA.json successfully written:")
print(json.dumps(metadata, indent=2))

METADATA.json successfully written:
{
  "dataset_name": "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
  "source_url": "https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset",
  "revision_sha": "430d1a89bd93bd1fa23c16f29dd53e73f0087443",
  "license": "cdla-sharing-1.0",
  "download_timestamp_utc": "2026-09-21T18:41:25.367896+00:00",
  "row_count": 26872,
  "column_count": 5,
  "columns": [
    "flags",
    "instruction",
    "category",
    "intent",
    "response"
  ],
  "artifacts": {
    "bitext_raw.parquet": {
      "sha256": "0c8ae53ede333a008aa821ee287c93a91d166b6f80cb81a3d665fa55d9a48782",
      "size_bytes": 3190236
    },
    "bitext_raw.csv": {
      "sha256": "6f81102b0100b97b8468eb04368033a23206bf1fde9d53500d5806ec1001a434",
      "size_bytes": 19202474
    }
  }
}


### 7. Inspect Schema, Data Types & Missing Values
Validate structure, column types, and check for any null values in the raw dataset.

In [7]:
print("Schema:")
for col, dtype in df.schema.items():
    print(f"  - {col}: {dtype}")

print("\nNull counts per column:")
null_counts = df.null_count()
print(null_counts)

Schema:
  - flags: String
  - instruction: String
  - category: String
  - intent: String
  - response: String

Null counts per column:
shape: (1, 5)
┌───────┬─────────────┬──────────┬────────┬──────────┐
│ flags ┆ instruction ┆ category ┆ intent ┆ response │
│ ---   ┆ ---         ┆ ---      ┆ ---    ┆ ---      │
│ u32   ┆ u32         ┆ u32      ┆ u32    ┆ u32      │
╞═══════╪═════════════╪══════════╪════════╪══════════╡
│ 0     ┆ 0           ┆ 0        ┆ 0      ┆ 0        │
└───────┴─────────────┴──────────┴────────┴──────────┘


### 8. Inspect 20 Random Rows
Examine sample instruction-response pairs across categories to understand the content.

In [8]:
sample_df = df.sample(n=20, seed=42)

for idx, row in enumerate(sample_df.iter_rows(named=True), start=1):
    print(f"=== Sample {idx:02d} ===")
    print(f"Flags:       {row['flags']}")
    print(f"Category:    {row['category']}")
    print(f"Intent:      {row['intent']}")
    print(f"Instruction: {row['instruction']}")
    print(f"Response:    {row['response']}")
    print("-" * 80)

=== Sample 01 ===
Flags:       BILQZ
Category:    ACCOUNT
Intent:      registration_problems
Instruction: how to reportr an error with signup
Response:    Thank you for reaching out to us regarding the error you encountered during the signup process. We're here to assist you in resolving this issue. To report the error, please follow these steps:

1. Go to our website and log in to your account.
2. Navigate to the "Help" or "Support" section of our website.
3. Look for the "Contact Us" or "Customer Support" option.
4. Click on the option to provide your feedback or report the error.
5. Fill out the form with the necessary details and describe the error you encountered during signup in as much detail as possible.
6. Submit the form or send the report.

Our support team will receive your report and work on resolving the issue as soon as possible. Should they require any additional information, they may reach out to you for further clarification. We appreciate your patience and cooperatio

### 9. Stage 1.1 Summary & Findings
- **Raw Row Count:** 26,872 rows, 5 columns (`flags`, `instruction`, `category`, `intent`, `response`).
- **Nulls:** 0 null values detected across all columns.
- **Provenance:** Pinned revision `430d1a89bd93bd1fa23c16f29dd53e73f0087443` recorded in `METADATA.json` with SHA-256 checksums.
- **Observations:** Instruction text contains templated placeholders like `{{Order Number}}` and variations of common customer intents.
- **Next Subtask:** Proceed to Stage 1.2 (`02_profile.ipynb`) for distribution analysis, length percentiles, and near-duplicate clustering.
